# Universal Document Ingestion for RAG with Pinecone

This notebook provides a complete pipeline for ingesting various document types (`.pdf`, `.docx`, `.xlsx`, `.txt`, `.xml`, `.jpg`, `.png`), processing their content, generating embeddings, and upserting them into a Pinecone vector index. It is designed to handle both text-based and scanned (image-based) documents automatically.

### 1. Setup: Install Dependencies

**Install Tesseract-OCR**

Tesseract is an optical character recognition (OCR) engine required for extracting text from images. Please install it on your system.

* **Windows**:
    Download and run the installer from the [Tesseract at UB Mannheim](https://github.com/UB-Mannheim/tesseract/wiki) page. During installation, make sure to note the installation path. You may need to add this path (e.g., `C:\Program Files\Tesseract-OCR`) to your system's `PATH` environment variable.

* **macOS**:
    You can use Homebrew to install Tesseract:
    ```bash
    brew install tesseract
    ```

* **Linux (Debian/Ubuntu)**:
    ```bash
    sudo apt-get update
    sudo apt-get install tesseract-ocr
    ```

### 2. Environment Configuration

**Create a `.env` file** in the same directory as this notebook. This file will store your secret API keys. Add the following content to it, replacing the placeholder values with your actual credentials:

```
PINECONE_API_KEY="YOUR_PINECONE_API_KEY_HERE"
PINECONE_INDEX_NAME="your-pinecone-index-name"
GEMINI_API_KEY="YOUR_GEMINI_API_KEY_HERE"
```

In [ ]:
import os
import io
from pathlib import Path
from typing import List
import xml.etree.ElementTree as ET
from dotenv import load_dotenv

# Data Handling & Processing
import fitz  # PyMuPDF
import docx
import pandas as pd
from PIL import Image
import pytesseract
from langchain.text_splitter import RecursiveCharacterTextSplitter
from tqdm.auto import tqdm

# AI & Vector DB
import torch
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
import google.generativeai as genai

# Load environment variables from .env file
load_dotenv()

# Load API keys and environment variables
try:
    PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
    PINECONE_INDEX_NAME = os.getenv('PINECONE_INDEX_NAME')
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

    if not all([PINECONE_API_KEY, PINECONE_INDEX_NAME, GEMINI_API_KEY]):
        raise ValueError("One or more environment variables (PINECONE_API_KEY, PINECONE_INDEX_NAME, GEMINI_API_KEY) are missing.")

    # Configure the Generative AI client
    genai.configure(api_key=GEMINI_API_KEY)
    print("API keys and environment variables loaded successfully.")

except Exception as e:
    print(f"Error loading environment variables or configuring API: {e}")

### 3. Initialize Pinecone Index and Embedding Model

In [ ]:
# --- Configuration ---
MODEL_NAME = "all-mpnet-base-v2"
EMBEDDING_DIMENSIONS = 768  # Fixed for the chosen model
PINECONE_CLOUD = 'aws' # Or 'gcp', 'azure'
PINECONE_REGION = 'us-east-1'

# --- Initialize Pinecone ---
try:
    print("Initializing Pinecone client...")
    pc = Pinecone(api_key=PINECONE_API_KEY)
    index_name = PINECONE_INDEX_NAME

    if index_name not in pc.list_indexes().names():
        print(f"Index '{index_name}' not found. Creating a new serverless index...")
        pc.create_index(
            name=index_name,
            dimension=EMBEDDING_DIMENSIONS,
            metric="cosine",
            spec=ServerlessSpec(
                cloud=PINECONE_CLOUD,
                region=PINECONE_REGION
            )
        )
        print(f"Index '{index_name}' created successfully.")
    else:
        print(f"Index '{index_name}' already exists.")

    index = pc.Index(index_name)
    print("\nInitial Pinecone index stats:")
    print(index.describe_index_stats())

except Exception as e:
    print(f"Error initializing Pinecone: {e}")

# --- Load Embedding Model (Optimized for GPU) ---
try:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"\nUsing device: {device}")
    print(f"Loading sentence transformer model: {MODEL_NAME}...")
    model = SentenceTransformer(MODEL_NAME, device=device)
    print("Embedding model loaded successfully.")
except Exception as e:
    print(f"Error loading embedding model: {e}")

### 4. Define Helper Functions for ETL (Extract, Transform, Load)

In [ ]:
def find_all_documents(folder_path: str) -> List[str]:
    """
    Recursively find all compatible documents in a folder and its subfolders.
    """
    supported_extensions = {
        '.pdf', '.docx', '.txt', '.xlsx',
        '.jpg', '.jpeg', '.png', '.xml'
    }
    document_paths = []

    folder = Path(folder_path)
    if not folder.exists():
        print(f"Error: The folder '{folder_path}' does not exist.")
        return document_paths

    for file_path in folder.rglob('*'):
        if (file_path.is_file() and
                file_path.suffix.lower() in supported_extensions and
                not file_path.name.startswith('._') and
                '__MACOSX' not in file_path.parts):
            document_paths.append(str(file_path))

    return sorted(document_paths)

def extract_text_from_document(file_path: str) -> List[dict]:
    """
    Extracts text from various document types using the best method for each.
    """
    file_extension = Path(file_path).suffix.lower()
    print(f"-> Extracting content from: {Path(file_path).name}")

    pages = []

    try:
        if file_extension == '.pdf':
            doc = fitz.open(file_path)
            for i, page in enumerate(tqdm(doc, desc="Processing PDF pages", leave=False)):
                text = page.get_text()
                # If text is minimal, assume it's a scanned page and run OCR
                if len(text.strip()) < 150:
                    try:
                        pix = page.get_pixmap(dpi=300)
                        img_data = pix.tobytes("png")
                        image = Image.open(io.BytesIO(img_data))
                        ocr_text = pytesseract.image_to_string(image)
                        text = ocr_text
                    except Exception as e:
                        print(f"   - OCR failed on page {i+1}: {e}, skipping page.")
                        continue
                if text:
                    pages.append({"page_number": i + 1, "content": text})

        elif file_extension == '.docx':
            doc = docx.Document(file_path)
            full_text = "\n".join([para.text for para in doc.paragraphs])
            if full_text:
                pages.append({"page_number": 1, "content": full_text})

        elif file_extension == '.txt':
            with open(file_path, 'r', encoding='utf-8') as f:
                full_text = f.read()
            if full_text:
                pages.append({"page_number": 1, "content": full_text})

        elif file_extension == '.xlsx':
            df_map = pd.read_excel(file_path, sheet_name=None, header=None)
            for i, (sheet_name, df) in enumerate(df_map.items()):
                sheet_text = df.to_string(index=False, header=False)
                cleaned_text = "\n".join([line for line in sheet_text.split('\n') if line.strip()])
                if cleaned_text:
                    content = f"--- Content from Sheet: {sheet_name} ---\n\n{cleaned_text}"
                    pages.append({"page_number": i + 1, "content": content})

        elif file_extension in ['.jpg', '.jpeg', '.png']:
            image = Image.open(file_path)
            ocr_text = pytesseract.image_to_string(image)
            if ocr_text:
                pages.append({"page_number": 1, "content": ocr_text})

        elif file_extension == '.xml':
            tree = ET.parse(file_path)
            root = tree.getroot()
            xml_text = " ".join(root.itertext())
            cleaned_text = " ".join(xml_text.split())
            if cleaned_text:
                pages.append({"page_number": 1, "content": cleaned_text})

        else:
            print(f"   - Warning: Unsupported file type '{file_extension}'. Skipping file.")
            return []

    except Exception as e:
        print(f"   - Error processing file {file_path}: {e}")
        return []

    print(f"   + Extracted {len(pages)} page(s) of content.")
    return pages

def chunk_document(pages: list[dict], chunk_size: int = 1000, chunk_overlap: int = 150) -> list[dict]:
    """Splits the document content from pages into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )

    chunks = []
    for page in pages:
        page_content = page.get('content', '')
        split_texts = text_splitter.split_text(page_content)
        for text in split_texts:
            chunks.append({
                "page_number": page.get('page_number'),
                "chunk_text": text
            })
    print(f"   + Document chunked into {len(chunks)} pieces.")
    return chunks

def process_and_upsert_document(doc_path: str, index, model, batch_size: int):
    """Performs the full ETL pipeline for a single document."""
    try:
        pages = extract_text_from_document(doc_path)
        if not pages:
            print(f"   - No content extracted from {doc_path}. Skipping.")
            return

        temp_chunks = chunk_document(pages)
        if not temp_chunks:
            print(f"   - No chunks were created from {doc_path}. Skipping.")
            return

        file_name = Path(doc_path).name
        chunks = []
        for i, chunk in enumerate(temp_chunks):
             chunks.append({
                "chunk_id": f"{file_name}_p{chunk['page_number']}_c{i}",
                "page_number": chunk['page_number'],
                "chunk_text": chunk['chunk_text']
             })

        print(f"   + Uploading {len(chunks)} chunks to Pinecone index '{index_name}'...")
        for i in tqdm(range(0, len(chunks), batch_size), desc=f"Upserting {file_name}", leave=False):
            i_end = min(i + batch_size, len(chunks))
            batch = chunks[i:i_end]

            ids = [chunk['chunk_id'] for chunk in batch]
            texts_to_embed = [chunk['chunk_text'] for chunk in batch]
            metadata = [
                {
                    'page_number': chunk['page_number'],
                    'text': chunk['chunk_text'],
                    'source_file': file_name
                 }
                for chunk in batch
            ]

            embeddings = model.encode(texts_to_embed, normalize_embeddings=True, show_progress_bar=False)
            vectors_to_upsert = list(zip(ids, embeddings.tolist(), metadata))
            index.upsert(vectors=vectors_to_upsert)

        print(f"--- Finished processing {file_name} ---")

    except Exception as e:
        print(f"An unexpected error occurred while processing {doc_path}: {e}")

### 5. Ingestion Process

This section runs the main processing loop to find all documents in your source folder and ingest them into Pinecone.

In [ ]:
# --- Configuration ---
# !!! IMPORTANT: Change this to the path where your documents are located !!!
SOURCE_FOLDER_PATH = './data'
BATCH_SIZE = 1024 # Adjust based on your available memory

# --- Main Execution Loop ---
all_docs = find_all_documents(SOURCE_FOLDER_PATH)

if not all_docs:
    print(f"\nNo documents found in '{SOURCE_FOLDER_PATH}'. Please check the folder path and file types.")
else:
    print(f"\nFound {len(all_docs)} documents to process:")
    for path in all_docs:
        print(f" - {Path(path).name}")

    # Optional: Uncomment the lines below if you want to clear all old data from the index first
    print("\nClearing all vectors from the index...")
    index.delete(delete_all=True)

    # Loop through each document and process it
    for doc_path in all_docs:
        process_and_upsert_document(doc_path, index, model, BATCH_SIZE)

    print("\n\n--- All Documents Processed! ---")
    print("\nFinal updated Pinecone index stats:")
    print(index.describe_index_stats())

### 6. RAG Querying and Answering

In [ ]:
# --- Initialize Gemini Model for Generation ---
try:
    generation_config = {
        "temperature": 0.2,
        "top_p": 1,
        "top_k": 5,
        "max_output_tokens": 2048,
    }
    # Note: Adjust the model name if you have access to other models like "gemini-1.5-pro-latest"
    gemini_model = genai.GenerativeModel(
        model_name="gemini-2.5-pro",
        generation_config=generation_config
    )
    print("Gemini model initialized successfully.")
except Exception as e:
    print(f"Error initializing Gemini model: {e}")

In [ ]:
def retrieve_context(query: str, top_k: int = 10) -> str:
    """Retrieves the most relevant context from Pinecone based on a user query."""
    print("Encoding query...")
    query_embedding = model.encode(query, normalize_embeddings=True).tolist()

    print("Querying Pinecone index...")
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    context_parts = []
    if results['matches']:
        print(f"Retrieved {len(results['matches'])} relevant chunks.")
        for match in results['matches']:
            if 'text' in match['metadata']:
                context_parts.append(match['metadata']['text'])
    else:
        print("No relevant context found in the vector database.")

    return "\n\n---\n\n".join(context_parts)

def generate_answer(query: str, context: str) -> str:
    """Generates an answer using the Gemini API based on the query and retrieved context."""
    if not context:
        return "I could not find any relevant information in the documents to answer your question."

    prompt = f"""You are a helpful and informative question-answering assistant.
    Your task is to answer the user's query based ONLY on the provided context.
    Do not use any external knowledge. Cite the source file if it is provided in the context.
    If the answer is not present in the context, state that you cannot answer the question with the given information.

    CONTEXT:
    ---
    {context}
    ---

    QUERY: {query}

    ANSWER:"""

    print("\nGenerating answer with Gemini...")
    try:
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"An error occurred while calling the Gemini API: {e}")
        return "Sorry, I encountered an error while trying to generate an answer."

def ask_question(query: str):
    """Full RAG pipeline: retrieves context, generates answer, and prints results."""
    retrieved_context = retrieve_context(query)

    if retrieved_context:
      print("\n--- Retrieved Context ---")
      print(retrieved_context)
      print("\n-------------------------")

    final_answer = generate_answer(query, retrieved_context)

    print("\n--- Final Answer ---")
    print(final_answer)
    print("--------------------\n")

In [ ]:
query = "when was the latest Annual General Meeting (AGM) held?"
ask_question(query)

In [ ]:
import csv
from datetime import datetime

In [ ]:
def short_ans(query:str):
    

In [ ]:
questions_to_process = [
    "Form Name",
    "Language",
    "Corporate Identity Number (CIN) of company",
    "Name of the company",
    "Address of registered office of the company",
    "e-mail ID of the company",
    "Authorised capital of the company as on the date of filing (in Rs.)",
    "Number of members of the company as on the date of filing",
    "Type of Industry",
    "Financial year to which financial statement relates: From",
    "Financial year to which financial statement relates: To",
    "Nature of financial statements",
    "Date of Board of Director's meeting in which financial statements are approved",
    "Whether provisional financial statements filed earlier",
    "Whether adopted in adjourned AGM",
    "Whether Annual General Meeting (AGM) held",
    "Date of AGM",
    "Due date of AGM",
    "Whether any extension for financial year or AGM granted",
    "Whether Schedule III of the Companies Act, 2013 is applicable",
    "Whether financial statements have been drawn on the basis of",
    "Whether consolidated Financial Statements are also being filed",
    "Whether CAG of India has commented upon or supplemented the audit report",
    "Whether CAG of India has conducted supplementary or test audit",
    "Whether Secretarial Audit is applicable",
    "Whether detailed disclosure with respect to Director's report Sec 134(3) is attached",
    "XBRL document in respect Consolidated financial statement: Status",
    "List of Attachment(s)",
    "Declaration Resolution number",
    "Declaration Dated",
    "Signatory Designation",
    "Director identification number",
    "Certifying Professional",
    "Professional Status",
    "Membership Number"
]

In [ ]:
wh_question_list = [
    "What is the name of the form being filed?",
    "What is the Corporate Identity Number (CIN) of the company?",
    "What is the legal name of the company?",
    "Where is the registered office of the company located?",
    "What is the official email ID of the company?",
    "What is the authorised capital of the company in Rupees as of the filing date?",
    "How many members does the company have?",
    "What type of industry does the company belong to?",
    "Which financial year do these financial statements cover?",
    "When did the financial year begin and end?",
    "What is the nature of the financial statements (e.g., standalone, consolidated)?",
    "When did the Board of Directors' meeting to approve the financial statements take place?",
    "When was the Annual General Meeting (AGM) held?",
    "What was the due date for the AGM?",
    "Which accounting standards were the financial statements based on?",
    "Who commented on or supplemented the audit report (e.g., CAG of India)?",
    "What disclosures from the Director's report are attached?",
    "What is the status of the XBRL document for the consolidated financial statements?",
    "What attachments are included with this filing?",
    "What is the declaration's resolution number?",
    "When was the declaration dated?",
    "Who signed this document on behalf of the company?",
    "What is the designation or title of the signatory?",
    "What is the Director Identification Number (DIN) of the signatory?",
    "Who is the professional that certified this document?",
    "What is the professional status of the certifier (e.g., Chartered Accountant, Company Secretary)?",
    "What is the membership number of the certifying professional?"
]

In [ ]:
# Test Excel OCR on a single file
import pandas as pd
from typing import List
from PIL import Image, ImageDraw

def find_all_documents(folder_path: str) -> List[str]:
    """
    Recursively find all compatible documents in a folder and its subfolders.
    """
    supported_extensions = {
        '.pdf', '.docx', '.txt', '.xlsx',
        '.jpg', '.jpeg', '.png', '.xml'
    }
    document_paths = []

    folder = Path(folder_path)
    if not folder.exists():
        print(f"Error: The folder '{folder_path}' does not exist.")
        return document_paths

    for file_path in folder.rglob('*'):
        if (file_path.is_file() and
                file_path.suffix.lower() in supported_extensions and
                not file_path.name.startswith('._') and
                '__MACOSX' not in file_path.parts):
            document_paths.append(str(file_path))

    return sorted(document_paths)

def excel_sheet_to_image(file_path: str, sheet_name: str, max_width: int = 2000, max_height: int = 3000) -> Image.Image:
    """
    Converts an Excel sheet to an image for OCR.
    Creates a visual representation of the spreadsheet.
    """
    try:
        # Read Excel sheet
        df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
        
        # Remove completely empty rows and columns
        df = df.dropna(how='all').dropna(axis=1, how='all')
        
        if df.empty:
            return None
        
        # Cell dimensions
        cell_width = 120
        cell_height = 30
        padding = 5
        
        # Calculate image size
        img_width = min(cell_width * len(df.columns), max_width)
        img_height = min(cell_height * len(df), max_height)
        
        # Create white background image
        img = Image.new('RGB', (img_width, img_height), color='white')
        draw = ImageDraw.Draw(img)
        
        # Draw cells and text
        for i, row in df.iterrows():
            if i * cell_height >= max_height:
                break
            for j, val in enumerate(row):
                if j * cell_width >= max_width:
                    break
                    
                x = j * cell_width
                y = i * cell_height
                
                # Draw cell border
                draw.rectangle([x, y, x + cell_width, y + cell_height], outline='black', width=1)
                
                # Draw text if not NaN
                if pd.notna(val):
                    text = str(val)[:15]  # Truncate long text
                    draw.text((x + padding, y + padding), text, fill='black')
        
        return img
    
    except Exception as e:
        print(f"      - Error converting sheet to image: {e}")
        return None

def extract_text_from_document(file_path: str) -> List[dict]:
    """
    Extracts text from various document types using the best method for each.
    Uses OCR for Excel files by converting sheets to images.
    """
    file_extension = Path(file_path).suffix.lower()
    print(f"-> Extracting content from: {Path(file_path).name}")

    pages = []

    try:
        if file_extension == '.pdf':
            doc = fitz.open(file_path)
            for i, page in enumerate(tqdm(doc, desc="Processing PDF pages", leave=False)):
                text = page.get_text()
                # If text is minimal, assume it's a scanned page and run OCR
                if len(text.strip()) < 150:
                    try:
                        pix = page.get_pixmap(dpi=300)
                        img_data = pix.tobytes("png")
                        image = Image.open(io.BytesIO(img_data))
                        ocr_text = pytesseract.image_to_string(image)
                        text = ocr_text
                    except Exception as e:
                        print(f"   - OCR failed on page {i+1}: {e}, skipping page.")
                        continue
                if text:
                    pages.append({"page_number": i + 1, "content": text})

        elif file_extension == '.docx':
            doc = docx.Document(file_path)
            full_text = "\n".join([para.text for para in doc.paragraphs])
            if full_text:
                pages.append({"page_number": 1, "content": full_text})

        elif file_extension == '.txt':
            with open(file_path, 'r', encoding='utf-8') as f:
                full_text = f.read()
            if full_text:
                pages.append({"page_number": 1, "content": full_text})

        elif file_extension == '.xlsx':
            # OCR approach for Excel files
            print(f"   - Using OCR for Excel file...")
            xl = pd.ExcelFile(file_path)
            
            for i, sheet_name in enumerate(tqdm(xl.sheet_names, desc="Processing Excel sheets", leave=False)):
                try:
                    # Convert sheet to image
                    sheet_img = excel_sheet_to_image(file_path, sheet_name)
                    
                    if sheet_img:
                        # Run OCR on the image
                        ocr_text = pytesseract.image_to_string(sheet_img, config='--psm 6')
                        
                        if ocr_text.strip():
                            content = f"=== Sheet: {sheet_name} ===\n\n{ocr_text}"
                            pages.append({"page_number": i + 1, "content": content})
                            print(f"      + Sheet '{sheet_name}' processed with OCR")
                except Exception as e:
                    print(f"      - Error processing sheet '{sheet_name}': {e}")
                    continue

        elif file_extension in ['.jpg', '.jpeg', '.png']:
            image = Image.open(file_path)
            ocr_text = pytesseract.image_to_string(image)
            if ocr_text:
                pages.append({"page_number": 1, "content": ocr_text})

        elif file_extension == '.xml':
            tree = ET.parse(file_path)
            root = tree.getroot()
            xml_text = " ".join(root.itertext())
            cleaned_text = " ".join(xml_text.split())
            if cleaned_text:
                pages.append({"page_number": 1, "content": cleaned_text})

        else:
            print(f"   - Warning: Unsupported file type '{file_extension}'. Skipping file.")
            return []

    except Exception as e:
        print(f"   - Error processing file {file_path}: {e}")
        return []

    print(f"   + Extracted {len(pages)} page(s) of content.")
    return pages

def chunk_document(pages: list[dict], chunk_size: int = 1000, chunk_overlap: int = 150) -> list[dict]:
    """Splits the document content from pages into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )

    chunks = []
    for page in pages:
        page_content = page.get('content', '')
        split_texts = text_splitter.split_text(page_content)
        for text in split_texts:
            chunks.append({
                "page_number": page.get('page_number'),
                "chunk_text": text
            })
    print(f"   + Document chunked into {len(chunks)} pieces.")
    return chunks

def process_and_upsert_document(doc_path: str, index, model, batch_size: int, namespace: str):
    """Performs the full ETL pipeline for a single document."""
    try:
        pages = extract_text_from_document(doc_path)
        if not pages:
            print(f"   - No content extracted from {doc_path}. Skipping.")
            return

        temp_chunks = chunk_document(pages)
        if not temp_chunks:
            print(f"   - No chunks were created from {doc_path}. Skipping.")
            return

        file_name = Path(doc_path).name
        chunks = []
        for i, chunk in enumerate(temp_chunks):
             chunks.append({
                "chunk_id": f"{file_name}_p{chunk['page_number']}_c{i}",
                "page_number": chunk['page_number'],
                "chunk_text": chunk['chunk_text']
             })

        print(f"   + Uploading {len(chunks)} chunks to namespace '{namespace}'...")
        for i in tqdm(range(0, len(chunks), batch_size), desc=f"Upserting {file_name}", leave=False):
            i_end = min(i + batch_size, len(chunks))
            batch = chunks[i:i_end]

            ids = [chunk['chunk_id'] for chunk in batch]
            texts_to_embed = [chunk['chunk_text'] for chunk in batch]
            metadata = [
                {
                    'page_number': chunk['page_number'],
                    'text': chunk['chunk_text'],
                    'source_file': file_name
                 }
                for chunk in batch
            ]

            embeddings = model.encode(texts_to_embed, normalize_embeddings=True, show_progress_bar=False)
            vectors_to_upsert = list(zip(ids, embeddings.tolist(), metadata))
            index.upsert(vectors=vectors_to_upsert, namespace=namespace)

        print(f"--- Finished processing {file_name} ---")

    except Exception as e:
        print(f"An unexpected error occurred while processing {doc_path}: {e}")
test_excel_file = "./data-v3.0_IndAs_Standalone-ExampleCo.xlsx"
import pytesseract

# Get sheet names
xl = pd.ExcelFile(test_excel_file)
print(f"\nTotal sheets: {len(xl.sheet_names)}")
print(f"Testing on first sheet: '{xl.sheet_names[4]}'\n")

# Convert first sheet to image
print("Step 1: Converting Excel sheet to image...")
sheet_img = excel_sheet_to_image(test_excel_file, xl.sheet_names[4])

if sheet_img:
    print(f"   ✓ Image created: {sheet_img.size[0]}x{sheet_img.size[1]} pixels")
    
    # Run OCR
    print("\nStep 2: Running OCR on image...")
    ocr_text = pytesseract.image_to_string(sheet_img, config='--psm 6')
    
    print(f"   ✓ OCR completed: {len(ocr_text)} characters extracted")
    
    # Show results
    print("\n" + "="*80)
    print("PROCESSED OUTPUT (first 1000 characters):")
    print("="*80)
    print(ocr_text[:1000])
    print("\n..." if len(ocr_text) > 1000 else "")
    
    print("\n" + "="*80)
    print(f"Full text length: {len(ocr_text)} characters")
    print(f"Lines extracted: {len(ocr_text.splitlines())}")
    print("="*80)
else:
    print("   ✗ Failed to create image from sheet")

In [ ]:
ocr_text